# AGO
---
## Requisitos
- Geração aleatória
- Seleção por proporcionalidade do fitness e por torneio 
- Cruzamento por 1 ponto e por 2 pontos - Podendo ajustar a taxa de cruzamento
- Mutação - Podendo ajustar a taxa de mutação
- Sphere: 30 dimensões (ou seja 30 variáveis) #parametrização
        - Os limites de cada variável é de: -100 a 100  #parametrização
- Tamanho da População = 30 
- Quantidade de gerações = 20 

---

- Cromossomo: É a estrutura de dados que representa uma solução completa para o problema. No código, cada indivíduo da lista pop será um cromossomo.
- Gene: É cada elemento individual dentro dessa estrutura (cada valor na lista). Se o problema tem 6 variáveis que precisam ser ajustadas para encontrar o melhor resultado, então o cromossomo terá 6 genes. Ou seja, na Sphere, seriam 6 dimensões

- Sobre os "var_x": são limites, definem o intervalo válido para o valor de cada gene. No caso da Sphere, cada variável (gene) deve estar entre -100 e 100: cromossomo = [45.3, -72.1, 0.5, 88.9, -12.4, ...]

In [542]:
import random

In [543]:
# parametros do AG
n_pop = 30
quantidade_genes = 30
n_geracoes = 25
pop = []
nova_pop = []
n_cortes = 2
taxa_crossover = 0.5242
taxa_mutacao = 0.0161


# Parametros do problema
n_dimensoes = quantidade_genes
var_min = -100
var_max = 100



## Gerar População

In [544]:
pop.clear()
#Loop para criar cada indivíduo da população:
for i in range(n_pop):
    cromossomo = [random.uniform(var_min, var_max) for _ in range(n_dimensoes)]
    pop.append(cromossomo)
    #print(f"cromossomo {i+1} gerado: {cromossomo}")
    #print("\n")

In [545]:
#pop

## Função Sphere

In [546]:
def sphere(x):
    """
    Sphere Function
    Mínimo global: f(0, ..., 0) = 0
    """
    return sum(xi**2 for xi in x)

## Avaliar cada individuo 

In [547]:
pop_aval= []
pop_aval.clear()
for cromossomo in pop:
    nota_aval = sphere(cromossomo)
    pop_aval.append((cromossomo, nota_aval))

#pop_aval
    

## Selecionar os individuos mais aptos

- Pais 1 com ROLETA
- lembrando: problema de minimização -> quanto menor a avaliação, maior o valor da aptidão.
- fitness = 1/(1 + nota avaliaçao)
- pop_aval = [(cromossomo, nota_aval), ...]

In [548]:
def calcular_fitness(pop_avaliada):
    """
    Recebe: lista de tuplas (cromossomo, valor_sphere)
    Retorna: lista de pesos de fitness
    """
    return [1 / (1 + individuo[1]) for individuo in pop_avaliada]

#def mostrar_probabilidade(pop_avaliada):
#    """
#    Mostra cada cromossomo com sua avaliação e probabilidade relativa
#    """
#    fitness = calcular_fitness(pop_avaliada)
#    total = sum(fitness)
#    probabilidades = [f/total for f in fitness]
#
#    for i, (crom, fitness) in enumerate(pop_avaliada):
#        print(f"Cromossomo {i+1}: avaliação={fitness:.2f}, probabilidade={probabilidades[i]*100:.2f}%")

def selecao_proporcional(pop_avaliada, n_pop_selec):
    """
    Recebe: população avaliada e quantidade a selecionar
    Retorna: apenas os cromossomos selecionados (sem o valor da sphere)
    """
    fitness = calcular_fitness(pop_avaliada)

    grupo_survivors = random.choices(pop_avaliada, weights=fitness, k=n_pop_selec)
    
    # retorna só os cromossomos, sem o valor da sphere
    return [individuo[0] for individuo in grupo_survivors]

In [549]:
#mostrar_probabilidade(pop_aval)
#len(p1_survivors)

In [550]:
def selecao_torneio(pop_avaliada, n_pop_selec):
    """
    Recebe: população avaliada [(cromossomo, avaliação), ...]
    Retorna: lista de cromossomos selecionados via torneio
    """
    grupo_survivors = []
    for rodada in range(n_pop_selec):
        # sorteiar 2 competidores
        c1, c2 = random.choices(pop_avaliada, k=2)

        # mostrar os competidores
        #print(f"Torneio {rodada+1}:")
        #print(f"  Competidor 1 -> avaliação={c1[1]:.2f}")
        #print(f"  Competidor 2 -> avaliação={c2[1]:.2f}")

        # comparar avaliações (menor é melhor)
        if c1[1] < c2[1]:
            survivor = c1
        else:
            survivor = c2

        #print(f"  Vencedor -> avaliação={survivor[1]:.2f}\n")

        grupo_survivors.append(survivor[0])  # pega só o cromossomo

    return grupo_survivors

In [551]:
p1_survivors = selecao_torneio(pop_aval, n_pop)
p2_survivors= selecao_torneio(pop_aval, n_pop)
len(p2_survivors)

30

## Criar novos individuos a partir dos pais sobreviventes/selecionados

In [552]:
def crossover_1pt(p1, p2, taxa_crossover, quantidade_genes):
    """
    Recebe: dois pais (Selecionados/sobreviventes), taxa de crossover e a quantidade de genes
    Retorna: dois filhos
    """
    # verifica se há cruzamento
    if random.random() < taxa_crossover:
        ponto_corte = random.randint(1, quantidade_genes - 1)
        
        # troca as partes após o ponto de corte
        filho1 = p1[:ponto_corte] + p2[ponto_corte:]
        filho2 = p2[:ponto_corte] + p1[ponto_corte:]
        #print(f"crossover: {filho1} + {filho2}\n")
    else:
        # sem cruzamento, filhos são cópias dos pais
        filho1 = p1[:]
        filho2 = p2[:]
        #print(f"sem crossover: {filho1} + {filho2}\n")
    
    return filho1, filho2

In [553]:
def mutacao(cromossomo, taxa_mutacao, quantidade_genes, var_min, var_max):
    """
    Recebe: cromossomo, quantidade de genes, taxa de mutação e limites
    Retorna: cromossomo (com possíveis mutações)
    """
    cromossomo_mutado = cromossomo[:]  # copia para não alterar o original
    
    for i in range(quantidade_genes):
        if random.random() < taxa_mutacao:
            cromossomo_mutado[i] = random.uniform(var_min, var_max)
            #print(f"mutacao no {i+1}: {cromossomo} ---> {cromossomo_mutado}\n")
    
    return cromossomo_mutado

In [554]:
def gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes,n_pop,
                          taxa_crossover, taxa_mutacao, 
                          var_min, var_max, nova_pop):
    """
    Recebe: listas de pais selecionados, quantidade de genes, a lista de nova populção, taxas e limites
    Retorna: nova população após crossover e mutação
    """
    
    # percorre os pais em pares
    for i in range(0, n_pop):
        
        p1 = random.choice(p1_survivors)
        p2 = random.choice(p2_survivors)
        
        # crossover
        filho1, filho2 = crossover_1pt(p1, p2, taxa_crossover, quantidade_genes)
        
        # mutação em cada filho
        filho1 = mutacao(filho1, taxa_mutacao, quantidade_genes, var_min, var_max)
        filho2 = mutacao(filho2, taxa_mutacao, quantidade_genes, var_min, var_max)
        
        nova_pop.append(filho1)
        nova_pop.append(filho2)
        
    return nova_pop

In [555]:
nova_pop.clear()
gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes, n_pop, taxa_crossover, taxa_mutacao, var_min, var_max, nova_pop)

[[70.69392138547147,
  54.94309132165088,
  -83.3341210824114,
  82.50089712727083,
  -73.16229399414284,
  24.584941005252105,
  2.2300687272269073,
  -54.95463353032706,
  3.581900695556257,
  51.98326094115899,
  46.27586874017268,
  -90.48335782953177,
  61.397008153686926,
  97.3163623028656,
  18.41502181516998,
  -2.976812292481611,
  -2.5938490813818333,
  -74.75552621732163,
  67.05611852595518,
  50.11445003470462,
  79.18275104329564,
  -67.30685077455917,
  10.490680079178645,
  2.6299456248027866,
  -18.710300176966783,
  -84.18852741023926,
  59.87747026823942,
  32.524618759995946,
  -89.2616391472462,
  -15.228666505364743],
 [-50.84500190984005,
  88.61692229306107,
  74.46629012082397,
  10.815423463291623,
  58.33942911187259,
  90.65986194328994,
  65.02857325498675,
  -14.570527409253927,
  43.92383530982465,
  -82.28137025541626,
  15.39772416327665,
  -10.38312100136028,
  61.13649395523893,
  -92.39207872070689,
  -62.28685196829944,
  -8.859855164930678,
  -46.

In [556]:
#len(nova_pop)

In [557]:
#print(nova_pop)

## Loop exec - Critério de parada - GERAÇÕES

In [558]:
historico_melhor = []  # para visualizar a evolução

for geracao in range(n_geracoes):

    # 1. Avaliar população atual
    pop_aval = [(crom, sphere(crom)) for crom in pop]

    # 2. Selecionar pais
    p1_survivors = selecao_proporcional(pop_aval, n_pop)
    p2_survivors = selecao_torneio(pop_aval, n_pop)

    # 3. Gerar 60 filhos
    nova_pop.clear()
    gerar_nova_pop(p1_survivors, p2_survivors, quantidade_genes, n_pop,
                   taxa_crossover, taxa_mutacao, var_min, var_max, nova_pop)

    # 4. Avaliar os 60 filhos e pegar os 30 melhores
    filhos_aval = [(crom, sphere(crom)) for crom in nova_pop]
    filhos_aval.sort(key=lambda x: x[1])          # ordena do menor para o maior
    pop = [ind[0] for ind in filhos_aval[:n_pop]]  # pega os 30 melhores

    # 5. Registrar o melhor dessa geração
    melhor_fitness = filhos_aval[0][1]
    historico_melhor.append(melhor_fitness)
    print(f"Geração {geracao+1:02d} | Melhor fitness: {melhor_fitness:.2f}")

# Resultado final
print(f"\n{'='*40}")
print(f"RESULTADO FINAL após {n_geracoes} gerações:")
print(f"Melhor fitness encontrado: {min(historico_melhor):.2f}")
print(f"Geração onde ocorreu: {historico_melhor.index(min(historico_melhor)) + 1}")

Geração 01 | Melhor fitness: 53374.49
Geração 02 | Melhor fitness: 53419.48
Geração 03 | Melhor fitness: 46310.18
Geração 04 | Melhor fitness: 45413.03
Geração 05 | Melhor fitness: 45247.74
Geração 06 | Melhor fitness: 42031.96
Geração 07 | Melhor fitness: 36870.69
Geração 08 | Melhor fitness: 35352.48
Geração 09 | Melhor fitness: 26603.65
Geração 10 | Melhor fitness: 30168.33
Geração 11 | Melhor fitness: 29026.39
Geração 12 | Melhor fitness: 27611.40
Geração 13 | Melhor fitness: 24492.50
Geração 14 | Melhor fitness: 22564.16
Geração 15 | Melhor fitness: 20605.30
Geração 16 | Melhor fitness: 19391.53
Geração 17 | Melhor fitness: 18144.03
Geração 18 | Melhor fitness: 17948.20
Geração 19 | Melhor fitness: 17416.56
Geração 20 | Melhor fitness: 16700.87
Geração 21 | Melhor fitness: 16600.76
Geração 22 | Melhor fitness: 14907.54
Geração 23 | Melhor fitness: 13852.21
Geração 24 | Melhor fitness: 12832.03
Geração 25 | Melhor fitness: 11718.88

RESULTADO FINAL após 25 gerações:
Melhor fitness 